# 06 – The nachtkritik archive: metadata, completeness, subcorpora

In session 4 we opened the nachtkritik files for the first time, turned them into tables and saw that the data are far from complete. This notebook is the first of a series in which we work with the archive step by step. Before we read a single text, we look at the **metadata**: what kinds of articles there are, when they appeared, who wrote them and about which productions, and above all **how complete** this information is.

From that we build **subcorpora**: well-defined parts of the archive that we can analyse and compare in the next sessions. Every subcorpus is a decision, and we want to know what we decide.

We work with the sample of 1000 files, which you already have on your Drive from session 4. The notebook is written so that the same code later runs on a larger sample

**Before we start**

The folder from session 4 must be in your Drive, here:

```
My Drive/
└── review-course/
    └── data/
        └── intern/
            └── 26-nachtkritik/
                └── cont-jsons-v3-sample-1000/
```

If Colab asks for access to your Drive, allow it

### 1. Load and organize

The first cell does everything we did in session 4 in one go, and a little more. It

1. reads every file of the sample into a list of dictionaries, `records`
2. separates the **articles** (files with a text) from the **pictures** (files without a text)
3. builds a table `articles` with **one row per article and one value per cell**: lists such as `articleAuthors` or `repTeam` are turned into single values (the first critic, the director, the size of the team, …)
4. adds a few columns that say whether a piece of information is there (`has_…`), and a readable name for every kind of article

Run it, then read it from top to bottom: every step is explained in a comment. Below we look at what came out

In [ ]:
from pathlib import Path
import json
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

# which sample? "100", "1000", or later a larger one; the rest of the notebook stays the same
SAMPLE = "1000"
FOLDER = Path("/content/drive/MyDrive/review-course/data/intern/26-nachtkritik") / ("cont-jsons-v3-sample-" + SAMPLE)

# 1. read every file into a dictionary, and collect the dictionaries in a list
records = []
for path in sorted(FOLDER.glob("*.json")):
    records.append(json.loads(path.read_text(encoding="utf-8")))

# the kinds of articles (convolute_id): nobody has written down what the letters mean,
# these names are our guesses from the titles (see session 4)
KINDS = {
    "T": "reviews", "M": "news", "P": "press review", "K": "review roundup",
    "F": "festivals", "D": "debates", "BL": "editorial blog", "TT": "Theatertreffen",
    "TB": "letters from abroad", "B": "books and films", "PP": "portraits and essays",
    "R": "miscellaneous", "S": "streams and specials", "U": "unsorted", "J": "annual survey",
    "UN": "internal pages",
}

# 2. and 3. one flat row per article; the pictures go into a list of their own
rows = []
pictures = []
for record in records:
    if "articlePlaintext" not in record:
        pictures.append(record)
    else:
        # record.get(key, default) gives the value of key, or the default when the key is missing
        authors = record.get("articleAuthors", [])
        places = record.get("articlePlaces", [])
        team = record.get("repTeam", [])

        row = {
            "article_id": record["article_id"],
            "kind": record["convolute_id"],
            "kind_name": KINDS.get(record["convolute_id"], "unknown"),
            "date": record["dateOfCreation"],                  # the day the article was put online
            "year": int(record["dateOfCreation"][:4]),
            "title": record["articleTitle"],
            "words": len(record["articlePlaintext"].split()),
            "comments": len(record.get("comments", [])),
            "n_critics": len(authors),
            "team_size": len(team),
        }

        # single values from the lists; where a list is empty, the cell stays empty (None)
        row["critic"] = authors[0] if len(authors) > 0 else None
        row["place"] = places[0] if len(places) > 0 else None
        row["production"] = record.get("repTitle")
        row["premiere"] = record.get("repKind")                # UA (world premiere) or DEA (first in German)

        row["director"] = None
        for role, name in team:
            if role == "Regie" and row["director"] is None:
                row["director"] = name

        rows.append(row)

articles = pd.DataFrame(rows)

# 4. is the information there? True or False for every article
articles["has_critic"] = articles["critic"].notna()
articles["has_place"] = articles["place"].notna()
articles["has_production"] = articles["team_size"] > 0

print("files:   ", len(records))
print("articles:", len(articles))
print("pictures:", len(pictures))

Two new things in the cell:

- `record.get("articleAuthors", [])` is the safe way to read a key that may be missing: instead of a `KeyError` we get the default, here an empty list. Without a default, `record.get("repTitle")` gives `None`, Python's word for *no value*
- `authors[0] if len(authors) > 0 else None` is an if-clause on one line: the first critic if there is one, otherwise `None`. pandas shows `None` as an empty cell

From now on we work almost only with the table `articles`. The original `records` stay in memory, in case we need something we have not taken over

In [ ]:
articles.head()

In [ ]:
# all columns, how many values each has, and of what type
articles.info()

In [ ]:
# a single article: loc[] picks a row by its index
articles.loc[0]

### 2. What is in the sample?

First an overview: which kinds of articles, from which years, how long? Every count here is a count of our **sample**, not of the archive: 1000 files picked at random from more than 40,000

In [ ]:
articles["kind_name"].value_counts()

In [ ]:
# value_counts(normalize=True) gives shares instead of numbers; round(2) keeps two decimals
articles["kind_name"].value_counts(normalize=True).round(2)

In [ ]:
articles["year"].value_counts().sort_index()

**groupby()** is the most useful new tool of this session. It splits the table into groups (here: one group per kind of article), computes something for every group, and puts the results together again.

Read it as: *group the articles by `kind_name`, take the column `words` of every group, and give me its median*. The median is the middle value: half of the articles are shorter, half are longer. Unlike the mean it is not pulled up by a few very long texts

In [ ]:
articles.groupby("kind_name")["words"].median().sort_values()

In [ ]:
# several numbers at once: agg() takes a list of what to compute
articles.groupby("kind_name")["words"].agg(["count", "median", "max"]).sort_values("count", ascending=False)

**Exercise 1**

1. Which kinds of articles have the most comments on average (`mean()`)? Which the fewest?
2. How many articles appeared per year **among the reviews only**?
3. Which critics wrote the most articles? And the most **reviews**?
4. Look at the five longest articles: `sort_values("words", ascending=False).head()`. What kind of articles are they?

<details>
<summary>Show a possible solution</summary>

```python
articles.groupby("kind_name")["comments"].mean().sort_values(ascending=False).round(1)

reviews_only = articles[articles["kind"] == "T"]
reviews_only["year"].value_counts().sort_index()

articles["critic"].value_counts().head(10)
reviews_only["critic"].value_counts().head(10)

articles.sort_values("words", ascending=False).head()[["kind_name", "year", "words", "title"]]
```

</details>

### 3. How complete is the metadata?

An analysis can only use what is there. If we want to compare critics, we need the critic; if we want to compare cities, the place; if we want to compare directors, the production data.

Because `True` counts as 1 and `False` as 0, the **mean** of a `has_…` column is the share of articles for which the information is there

In [ ]:
articles[["has_critic", "has_place", "has_production"]].mean().round(2)

In [ ]:
# the same per kind of article
articles.groupby("kind_name")[["has_critic", "has_place", "has_production"]].mean().round(2)

The kinds of articles differ a lot. News (`news`) almost never name a critic or a place; the press review does not review a production. That is not a mistake, it lies in the nature of these texts.

But even among the reviews, where we would expect everything, not everything is there. To see which combinations occur, we give every article a short **profile**: one letter for every piece of information that is there, a `-` for every gap. `CPR` means critic, place and production; `--R` only the production

In [ ]:
# map() replaces every value of a column with the value the dictionary gives for it
articles["profile"] = (
    articles["has_critic"].map({True: "C", False: "-"})
    + articles["has_place"].map({True: "P", False: "-"})
    + articles["has_production"].map({True: "R", False: "-"})
)
articles[["kind_name", "critic", "place", "production", "profile"]].head(10)

In [ ]:
articles["profile"].value_counts()

In [ ]:
# crosstab() counts the combinations of two columns: here, profiles per kind of article
pd.crosstab(articles["kind_name"], articles["profile"])

Profiles are precise, but there are many of them. For an overview we also count **how many** of the three pieces of information an article has, from 0 to 3. Adding up `True` and `False` works as with the mean: `True` counts as 1

In [ ]:
articles["completeness"] = articles["has_critic"].astype(int) + articles["has_place"].astype(int) + articles["has_production"].astype(int)
articles["completeness"].value_counts().sort_index()

Now the decisive question for our work with reviews: **is the completeness the same over the years?** If it is not, every comparison over time is in danger, because we would be comparing different kinds of data

In [ ]:
reviews = articles[articles["kind"] == "T"]

# the share of reviews with critic, place and production, per year
reviews.groupby("year")[["has_critic", "has_place", "has_production"]].mean().round(2)

From 2022 onwards, critic and place are missing in **every** review, while the production data are there. The reviews themselves have critics, of course, and they are printed on the website: the information got lost on its way into the export, perhaps because the website changed its layout and the export did not change with it.

This is the most important result of this section. Without it we would conclude that nachtkritik stopped working with critics in 2022, or that Berlin disappeared from the reviews. Anyone who counts critics or places over time must either stop in 2021, or recover the missing information from somewhere else (the text itself, the website)

**Exercise 2**

1. Which profiles occur among the reviews (`reviews["profile"]`)? How many reviews have no production data at all? Look at a few of their titles: why might the production be missing?
2. Among the reviews **with** production data: is there always a director? (hint: `reviews[reviews["has_production"]]["director"].isna().sum()`)
3. Does the date of creation (`dateOfCreation`, which we use) always match the date the article gives itself (`articleDate`)? Count the differences with a loop over `records` (careful: not every record has an `articleDate`)

<details>
<summary>Show a possible solution</summary>

```python
reviews["profile"].value_counts()

no_production = reviews[reviews["has_production"] == False]
print(len(no_production))
no_production["title"].head(10)
# special formats (theatre in prison, film trailers), guest performances, or the credits were not found in the text

with_production = reviews[reviews["has_production"]]
print(with_production["director"].isna().sum())
# some teams have no "Regie": "Inszenierung", "Konzept", collectives, or roles that went wrong in the export

same = 0
different = 0
for record in records:
    if "articlePlaintext" in record and "articleDate" in record:
        if record["articleDate"] == record["dateOfCreation"]:
            same = same + 1
        else:
            different = different + 1
print(same, different)
# about 60 differences: most by one or two days, in both directions (review written at night, put online
# the next day, or the other way round), a few by weeks or years, and two "dates" that are just the letter "e"
```

</details>

### 4. Building subcorpora

A **subcorpus** is a part of the archive that we define for a question. It is defined by conditions on the metadata, and the conditions should be written down, so that others (and we, in four weeks) know what is in it.

We keep our subcorpora together in a dictionary: the key is a name, the value the table

In [ ]:
subcorpora = {}

# all reviews
subcorpora["reviews"] = articles[articles["kind"] == "T"]

# reviews for which we know critic, place and production
subcorpora["reviews_complete"] = articles[(articles["kind"] == "T") & (articles["completeness"] == 3)]

# reviews before and since the break in the export
subcorpora["reviews_until_2021"] = articles[(articles["kind"] == "T") & (articles["year"] <= 2021)]
subcorpora["reviews_since_2022"] = articles[(articles["kind"] == "T") & (articles["year"] >= 2022)]

# reviews from Berlin and from everywhere else (only where we know the place)
subcorpora["reviews_berlin"] = articles[(articles["kind"] == "T") & (articles["place"] == "Berlin")]
subcorpora["reviews_elsewhere"] = articles[(articles["kind"] == "T") & articles["has_place"] & (articles["place"] != "Berlin")]

# everything that is not a review, but written by someone and longer than 300 words
subcorpora["other_texts"] = articles[(articles["kind"] != "T") & articles["has_critic"] & (articles["words"] > 300)]

Several conditions are combined with `&` (and) and `|` (or), and every condition needs its own round brackets.

Now an overview: how large is every subcorpus, which years does it cover, how long are its texts? We build it with the same pattern as always: an empty list, a loop, one dictionary per row

In [ ]:
overview_rows = []
for name in subcorpora:
    sub = subcorpora[name]
    overview_rows.append({
        "subcorpus": name,
        "articles": len(sub),
        "from": sub["year"].min(),
        "to": sub["year"].max(),
        "median_words": sub["words"].median(),
        "critics": sub["critic"].nunique(),       # nunique() counts the different values
    })

overview = pd.DataFrame(overview_rows)
overview

Look at the numbers before you use a subcorpus: a few dozen reviews are enough to read closely, but not to count reliably. `reviews_berlin` is small in our sample; in the whole archive it would be large.

To use the subcorpora in later sessions, we save the `article_id`s of each one. With them we can always find the texts again in `records`, or in a larger sample

In [ ]:
OUT = Path("/content/drive/MyDrive/review-course/data/intern/subcorpora-" + SAMPLE)
OUT.mkdir(exist_ok=True)

for name in subcorpora:
    subcorpora[name][["article_id", "kind", "year", "title"]].to_csv(OUT / (name + ".csv"), index=False)

print(sorted(p.name for p in OUT.glob("*.csv")))

**Exercise 3**

Define two subcorpora of your own, add them to `subcorpora`, and run the overview cell again. Some ideas:

- reviews of world premieres (`premiere == "UA"`)
- reviews with many comments (more than 10)
- the festival articles together with the Theatertreffen articles (hint: `articles["kind"].isin(["F", "TT"])`)
- the reviews of one critic, or of one director

Write down in a comment for each one: what question could it answer, and why is it (still) too small or not?

<details>
<summary>Show a possible solution</summary>

```python
# world premieres: are new plays reviewed differently? (very small: repKind is filled in for only 11 reviews, all from 2008 to 2013)
subcorpora["reviews_world_premieres"] = articles[(articles["kind"] == "T") & (articles["premiere"] == "UA")]

# much discussed reviews: what makes readers comment?
subcorpora["reviews_discussed"] = articles[(articles["kind"] == "T") & (articles["comments"] > 10)]

# festivals and Theatertreffen: the "event" side of the archive
subcorpora["festivals"] = articles[articles["kind"].isin(["F", "TT"])]
```

</details>

### 5. Visualizing metadata

Tables are exact, charts show patterns. pandas draws charts with `plot()`, using the library matplotlib in the background.

Some rules we follow:
- one chart answers one question, and its title says which
- a chart with one series gets one color; several series get a legend
- before interpreting a chart, ask how much data it rests on

We define our colors once, so that every chart looks the same

In [ ]:
# matplotlib draws the charts for pandas; we only need it for plt.show(), which finishes a chart:
# without it, several charts in one cell are drawn on top of each other
import matplotlib.pyplot as plt

BLUE = "#2a78d6"                                              # for charts with a single series
SHADES = ["#b7d3f6", "#6da7ec", "#2a78d6", "#104281"]          # one hue, light to dark: for an order (0 to 3)
KIND_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b4b2a9"]   # different hues: for categories, grey for "other"

**What kinds of articles appear over the years?**

`crosstab()` gives a table with one row per year and one column per kind; `plot(kind="bar", stacked=True)` puts the columns on top of each other. With sixteen kinds the chart would be unreadable, so we keep the four largest and put everything else under `other`

In [ ]:
top_kinds = articles["kind_name"].value_counts().index[:4]
articles["kind_group"] = articles["kind_name"].where(articles["kind_name"].isin(top_kinds), "other")
# where() keeps a value where the condition is True, and puts "other" everywhere else

per_year = pd.crosstab(articles["year"], articles["kind_group"])
per_year = per_year[list(top_kinds) + ["other"]]          # a fixed order of the columns: the largest first, "other" last

ax = per_year.plot(kind="bar", stacked=True, color=KIND_COLORS, edgecolor="white", width=0.8,
                   figsize=(10, 4), title="Articles per year and kind (sample)", xlabel="", ylabel="articles")

# plot() gives back the chart (ax); we use it to move the legend to the right of the bars
ax.legend(title="kind", loc="upper left", bbox_to_anchor=(1, 1))

**How complete are the reviews, year by year?**

The same kind of chart, now for the reviews only and by completeness (0 to 3). Here the colors stand for an order, so they are shades of one color

In [ ]:
reviews = subcorpora["reviews"]
per_year = pd.crosstab(reviews["year"], reviews["completeness"])

ax = per_year.plot(kind="bar", stacked=True, color=SHADES, edgecolor="white", width=0.8,
                   figsize=(10, 4), title="Reviews per year, by number of known fields (critic, place, production)",
                   xlabel="", ylabel="reviews")
ax.legend(title="known fields", loc="upper left", bbox_to_anchor=(1, 1))

In [ ]:
# the same as shares: a line for every piece of information
shares = reviews.groupby("year")[["has_critic", "has_place", "has_production"]].mean()
# xticks says which years to write on the axis: range(2007, 2026, 2) is every second year from 2007 to 2025
shares.plot(color=KIND_COLORS[:3], figsize=(10, 4), ylim=(0, 1.05), linewidth=2, marker="o",
            xticks=range(2007, 2026, 2), title="Share of reviews with this information", xlabel="", ylabel="share of reviews")

**Where are the reviewed productions?**

A horizontal bar chart (`barh`) leaves room for the names. `sort_values()` puts the largest bar on top

In [ ]:
places = subcorpora["reviews_until_2021"]["place"].value_counts().head(15).sort_values()
places.plot(kind="barh", color=BLUE, figsize=(6, 5), title="Places of reviewed productions, 2007–2021 (sample)", xlabel="reviews")

**How long are the texts?**

A histogram (`hist`) cuts the range of values into bins and counts the articles in every bin: it shows the distribution, not just the median

In [ ]:
subcorpora["reviews"]["words"].plot(kind="hist", bins=30, color=BLUE, edgecolor="white", figsize=(8, 4),
                                    title="Length of reviews (sample)", xlabel="words")

In [ ]:
# a boxplot compares distributions: the box holds the middle half of the values, the line in it is the median
compare = articles[articles["kind_group"] != "other"]
ax = compare.boxplot(column="words", by="kind_group", figsize=(8, 4), grid=False)

# boxplot() writes a title of its own above the chart; we replace it with ours
ax.figure.suptitle("")
ax.set_title("Length of articles by kind (sample)")
ax.set_xlabel("")
ax.set_ylabel("words")

**Exercise 4**

1. Draw the number of comments per year for the reviews (mean per year, a line chart). Does anything change? How many reviews per year does each point rest on?
2. Draw the ten critics with the most reviews as a horizontal bar chart
3. Compare the length of the reviews `until_2021` and `since_2022` with two medians, or two histograms. Did the reviews get shorter or longer?
4. Choose one of your own subcorpora from exercise 3 and draw one chart that describes it

<details>
<summary>Show a possible solution</summary>

```python
reviews.groupby("year")["comments"].mean().plot(color=BLUE, linewidth=2, marker="o", figsize=(10, 4),
                                                  xticks=range(2007, 2026, 2), title="Comments per review (mean)", xlabel="")
plt.show()
print(reviews["year"].value_counts().sort_index())   # often only 10 to 15 reviews per year: one discussed review moves the line

reviews["critic"].value_counts().head(10).sort_values().plot(kind="barh", color=BLUE, figsize=(6, 4),
                                                            title="Critics with the most reviews (sample)", xlabel="reviews")
plt.show()

print(subcorpora["reviews_until_2021"]["words"].median())
print(subcorpora["reviews_since_2022"]["words"].median())
```

</details>

### Summary

| step | what we did | tools |
|---|---|---|
| load and organize | files → `records` → one flat row per article in `articles` | `glob()`, `json.loads()`, `record.get()`, `pd.DataFrame()` |
| overview | kinds, years, lengths | `value_counts()`, `groupby()`, `median()`, `agg()` |
| completeness | shares, profiles, completeness over time | `mean()` of True/False, `map()`, `crosstab()` |
| subcorpora | named parts of the archive, written down as conditions | `&`, `|`, `isin()`, a dictionary of tables, `to_csv()` |
| visualizing | stacked bars, lines, horizontal bars, histograms, boxplots | `plot(kind=…)`, `boxplot()` |

What we learned about the archive:

- the export holds articles of very different kinds, and only the reviews (`T`) describe a production
- the metadata depend on the kind of article, and **they change over time**: since 2022 the export has lost critics and places
- every subcorpus is a decision about what counts; the sizes in the sample show what we can and cannot count reliably

In the next notebook we turn from the metadata to the texts of our subcorpora